In [ ]:


# Step 1 - Install required libraries
!pip install diffusers transformers accelerate torch --quiet

# Step 2 - Imports
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from diffusers import StableDiffusionPipeline

# 1. Load Pipeline
MODEL_ID = "runwayml/stable-diffusion-v1-5"

dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device  : {device}")
print(f"Loading : {MODEL_ID} ...")

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    safety_checker=None,
)
pipe = pipe.to(device)

if device == "cuda":
    pipe.enable_attention_slicing()

print("Pipeline ready!\n")

# 2. Prompts
prompts = [
    "A futuristic city skyline at sunset with flying cars, ultra-realistic, 4K",
    "A serene Japanese zen garden in spring with cherry blossoms, watercolor painting style",
    "A majestic lion wearing a crown in a fantasy forest, digital art, vibrant colors",
]

negative_prompt = "blurry, low quality, distorted, ugly, bad anatomy, watermark"

GEN_PARAMS = dict(
    num_inference_steps = 30,
    guidance_scale      = 7.5,
    height              = 512,
    width               = 512,
    negative_prompt     = negative_prompt,
    generator           = torch.Generator(device=device).manual_seed(42),
)

# 3. Generate Images
print("Generating images ...")
images = []
for i, prompt in enumerate(prompts, 1):
    print(f"  [{i}/{len(prompts)}] {prompt[:70]}...")
    result = pipe(prompt, **GEN_PARAMS)
    images.append((prompt, result.images[0]))

print("\nAll images generated!")

# 4. Display Images
fig = plt.figure(figsize=(18, 7))
gs  = gridspec.GridSpec(1, len(images), figure=fig, wspace=0.05)

for idx, (prompt, img) in enumerate(images):
    ax = fig.add_subplot(gs[idx])
    ax.imshow(img)
    ax.axis("off")
    wrapped = "\n".join([prompt[i:i+40] for i in range(0, len(prompt), 40)])
    ax.set_title(wrapped, fontsize=9, pad=6)

plt.suptitle("Stable Diffusion v1.5 — Generated Images",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("generated_images.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: generated_images.png")